### IMPORTS

In [1]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
scaler = StandardScaler()

### DATASETS

In [2]:
auto_mpg = pd.read_csv("auto_mpg_cleaned.csv")
auto_mpg = auto_mpg.dropna()
house_price = pd.read_csv('housing.csv')
house_price = house_price.dropna()
house_price = pd.get_dummies(house_price, dtype = int, drop_first = True)
insurance = pd.read_csv('insurance_cat2num.csv')

### CLASS SETUP

In [4]:
#One Hidden Layer Neural Network
        
'''======================================================================================'''

class OneHiddenLayerNNLeakyRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(OneHiddenLayerNNLeakyRelu, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # Input to hidden layer
        self.leaky_relu = nn.LeakyReLU()              # Activation function
        self.fc2 = nn.Linear(hidden_size, output_size) # Hidden to output layer
        self.batch_size = 32
        self.patience = 25
        self.min_delta = 1e-4
        
    def forward(self, x):
        out = self.fc1(x)   # Linear transformation
        out = self.leaky_relu(out) # Non-linearity
        out = self.fc2(out)  # Linear transformation to output
        return out
    def trainLeakyRelu(self, X_train, y_train, max_epochs=200, learning_rate=0.01):
        self.train()  # Set the model to training mode
        criterion = nn.MSELoss()  # Mean Squared Error Loss
        optimizer = torch.optim.SGD(self.parameters(), lr=learning_rate)  # Stochastic Gradient Descent

        batch_size = self.batch_size

        dataset = TensorDataset(X_train, y_train)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(max_epochs):
            epoch_loss = 0.0
            
            for batch_X, batch_y in dataloader:
            
                outputs = self(batch_X)  # Forward pass
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                epoch_loss+=loss.item()
            avg_loss = epoch_loss / len(dataloader)
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}/{max_epochs} | Avg Loss: {avg_loss:.4f}")

            if best_loss - avg_loss > self.min_delta:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= self.patience:
                print(f"\nEarly stopping triggered at Epoch {epoch+1}!")
                print(f"Loss hasn't improved by more than {self.min_delta} for {self.patience} consecutive epochs.")
                break
            
        print("Training Complete!")
    def testLeakyRelu(self, X_test, y_test):
        self.eval()  # Set the model to evaluation mode
        loss_fn = nn.MSELoss()
        with torch.no_grad():  # No need to compute gradients during testing
            outputs = self(X_test)  # Forward pass
            test_loss = loss_fn(outputs, y_test)
        print(f"Test Loss: {test_loss.item():.4f}")
        self.train()   
        return(outputs, test_loss.item())

### QOF VARIABLES

In [6]:
def get_qof(y_actual, y_pred, k, mod=None):
    """
    Calculates and compiles a comprehensive set of Quality of Fit (QoF) metrics 
    for a regression model.
    
    This function attempts to extract pre-calculated metrics from a fitted model 
    object (e.g., a Statsmodels result instance) if provided. If the object is not 
    provided or lacks specific metrics, it computes them manually using NumPy.

    Args:
        y_actual (array-like): The actual/true target values.
        y_pred (array-like): The predicted values generated by the model.
        k (int): The number of predictor variables (features) used in the model.
        mod (object, optional): A fitted model object (e.g., from Statsmodels) 
                                that may contain pre-calculated metrics. Defaults to None.

    Returns:
        list: A list containing 15 QoF metrics in the following specific order:
              [R^2, Adj R^2, SST, SSE, SDE, MSE, RMSE, MAE, sMAPE, 
               Number of Observations, DF Model, DF Residuals, F-statistic, AIC, BIC]
    """
    # ==========================================
    # --- Basic Parameters ---
    # ==========================================
    # Get the number of observations (m) in the original dataset
    m = len(y_actual)  

    # ==========================================
    # --- Sum of Squares ---
    # ==========================================
    # Calculate SSE (Sum of Squared Errors)
    sse = getattr(mod, "ssr", np.sum((y_actual - y_pred)**2))

    # Calculate SST (Total Sum of Squares)
    sst = getattr(mod, "centered_tss", np.sum((y_actual - np.mean(y_actual))**2))

    # ==========================================
    # --- R-Squared Metrics ---
    # ==========================================
    # Compute R-squared (Coefficient of Determination)
    r_squared = getattr(mod, "rsquared", 1 - (sse / sst))

    # Calculate Adjusted R-squared (penalizes for adding non-useful predictors)
    adj_r_squared = getattr(mod, "rsquared_adj", 1 - (1 - r_squared) * (m - 1) / (m - k))

    # ==========================================
    # --- Error Metrics ---
    # ==========================================
    # Standard Deviation of the Errors (SDE) / Residual Standard Error
    sde = np.sqrt(getattr(mod, "mse_resid", sse / (m - k)))

    # Calculate Mean Squared Error (MSE)
    mse = getattr(mod, "mse_resid", sse / m)

    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(getattr(mod, "mse_resid", mse))

    # Mean Absolute Error (MAE)
    mae = getattr(mod, "mae", np.mean(np.abs(y_actual - y_pred)))

    # Symmetric Mean Absolute Percentage Error (sMAPE)
    smape = getattr(mod, "smape", np.mean(2 * np.abs(y_actual - y_pred) / (np.abs(y_actual) + np.abs(y_pred))) * 100)

    # ==========================================
    # --- Statistical Tests & Degrees of Freedom ---
    # ==========================================
    # Calculate F-statistic for overall model significance
    f_stat = getattr(mod, "fvalue", ((sst - sse) / k) / (sse / (m - k)))

    # Get the degrees of freedom for the model (typically equivalent to k)
    df_model = getattr(mod, "df_model", k)

    # Get the degrees of freedom for the residuals (typically m - k)
    df_resid = getattr(mod, "df_resid", m - k)

    # ==========================================
    # --- Information Criteria ---
    # ==========================================
    # Calculate Akaike Information Criterion (AIC)
    aic = getattr(mod, "aic", m * np.log(mse) + 2 * k)

    # Calculate Bayesian Information Criterion (BIC)
    bic = getattr(mod, "bic", m * np.log(mse) + k * np.log(m))

    # ==========================================
    # --- Compile Results ---
    # ==========================================
    # Initialize a list to store exactly 15 elements
    qof = list(range(15))

    qof[0]  = r_squared
    qof[1]  = adj_r_squared
    qof[2]  = sst
    qof[3]  = sse
    qof[4]  = sde
    qof[5]  = mse
    qof[6]  = rmse
    qof[7]  = mae
    qof[8]  = smape
    qof[9]  = m
    qof[10] = df_model
    qof[11] = df_resid
    qof[12] = f_stat
    qof[13] = aic
    qof[14] = bic

    return qof

## IN-SAMPLE

### AUTOMPG

### HOUSE PRICES

### INSURANCE

## TRAIN TEST SPLIT

### AUTOMPG

In [7]:
X = auto_mpg.drop('mpg', axis = 1)
y = auto_mpg[['mpg']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32)
model_sigmoid = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_sigmoid.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_sigmoid.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 8.6134
Epoch 20/200 | Avg Loss: 8.1433
Epoch 30/200 | Avg Loss: 9.1871
Epoch 40/200 | Avg Loss: 6.7716
Epoch 50/200 | Avg Loss: 7.1330
Epoch 60/200 | Avg Loss: 7.6406
Epoch 70/200 | Avg Loss: 7.1528
Epoch 80/200 | Avg Loss: 6.7251
Epoch 90/200 | Avg Loss: 6.1224
Epoch 100/200 | Avg Loss: 6.3196
Epoch 110/200 | Avg Loss: 6.0292
Epoch 120/200 | Avg Loss: 5.8600
Epoch 130/200 | Avg Loss: 6.4924
Epoch 140/200 | Avg Loss: 6.3234
Epoch 150/200 | Avg Loss: 6.6753
Epoch 160/200 | Avg Loss: 5.9012

Early stopping triggered at Epoch 162!
Loss hasn't improved by more than 0.0001 for 25 consecutive epochs.
Training Complete!
Test Loss: 5.3284
[np.float64(0.8956038628712889), np.float64(0.8869041847772297), np.float64(4032.206075949367), np.float64(420.94673843603204), np.float64(2.417950057027279), np.float64(5.32843972703838), np.float64(2.308341336769409), np.float64(1.631583872927895), np.float64(7.165731142337546), 79, 7, 72, np.float64(88.24009872432961), np.float64(1

### HOUSE PRICES

In [8]:
X = house_price.drop('median_house_value', axis = 1)
y = house_price[['median_house_value']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = y_train / 100000
y_test_scaled = y_test / 100000
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled.to_numpy(), dtype=torch.float32)
model_sigmoid = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_sigmoid.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_sigmoid.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test_scaled.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 0.3411
Epoch 20/200 | Avg Loss: 0.3191
Epoch 30/200 | Avg Loss: 0.3069
Epoch 40/200 | Avg Loss: 0.2988
Epoch 50/200 | Avg Loss: 0.2928
Epoch 60/200 | Avg Loss: 0.2876
Epoch 70/200 | Avg Loss: 0.2821
Epoch 80/200 | Avg Loss: 0.2798
Epoch 90/200 | Avg Loss: 0.2750
Epoch 100/200 | Avg Loss: 0.2742
Epoch 110/200 | Avg Loss: 0.2695
Epoch 120/200 | Avg Loss: 0.2681
Epoch 130/200 | Avg Loss: 0.2663
Epoch 140/200 | Avg Loss: 0.2654
Epoch 150/200 | Avg Loss: 0.2629
Epoch 160/200 | Avg Loss: 0.2587
Epoch 170/200 | Avg Loss: 0.2605
Epoch 180/200 | Avg Loss: 0.2595
Epoch 190/200 | Avg Loss: 0.2616
Epoch 200/200 | Avg Loss: 0.2586
Training Complete!
Test Loss: 0.2937
[np.float64(0.7852001177598685), np.float64(0.784620289856889), np.float64(5589.046387258101), np.float64(1200.5265058176724), np.float64(0.5427777920005565), np.float64(0.2937427222455768), np.float64(0.5419803707198045), np.float64(0.3674127300448192), np.float64(18.87423848510443), 4087, 12, 4075, np.float64

### INSURANCE

In [9]:
X = insurance.drop('charges', axis = 1)
y = insurance[['charges']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = y_train / 6000
y_test_scaled = y_test / 6000
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled.to_numpy(), dtype=torch.float32)
model_sigmoid = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_sigmoid.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_sigmoid.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test_scaled.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 0.7335
Epoch 20/200 | Avg Loss: 0.6527
Epoch 30/200 | Avg Loss: 0.6130
Epoch 40/200 | Avg Loss: 0.5951
Epoch 50/200 | Avg Loss: 0.5807
Epoch 60/200 | Avg Loss: 0.5960
Epoch 70/200 | Avg Loss: 0.5741
Epoch 80/200 | Avg Loss: 0.5596
Epoch 90/200 | Avg Loss: 0.5517
Epoch 100/200 | Avg Loss: 0.5527
Epoch 110/200 | Avg Loss: 0.5485
Epoch 120/200 | Avg Loss: 0.5512
Epoch 130/200 | Avg Loss: 0.5261
Epoch 140/200 | Avg Loss: 0.5214
Epoch 150/200 | Avg Loss: 0.5256
Epoch 160/200 | Avg Loss: 0.5193
Epoch 170/200 | Avg Loss: 0.5232
Epoch 180/200 | Avg Loss: 0.5040
Epoch 190/200 | Avg Loss: 0.5171
Epoch 200/200 | Avg Loss: 0.5064
Training Complete!
Test Loss: 0.6168
[np.float64(0.8569622427008478), np.float64(0.8525440880352368), np.float64(1155.7405566609807), np.float64(165.31453724446047), np.float64(0.798924318431307), np.float64(0.6168452882255988), np.float64(0.7853949886685035), np.float64(0.4578896774811124), np.float64(27.61697565747082), 268, 9, 259, np.float64(1